# 本文件展示了LGflow与torch在链式求导以及梯度更新时的对比

In [1]:
from torch import nn as t_nn
from torch import optim as t_optim
import torch

In [2]:
from LG_flow import nn as l_nn
from LG_flow import optim as l_optim
import LG_flow

# 一. 创建模型

## 1.1 创建torch模型
    创建一个两层全连接的torch模型。

In [3]:
class TNet(t_nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = t_nn.Linear(5, 4)
        self.norm = t_nn.LayerNorm(4) # 不要问为什么是LayerNorm，这里只是测试不同。
        self.fc2 = t_nn.Linear(4, 3)

    def forward(self, x):
        x = self.fc1(x)
        x = self.norm(x)
        x = self.fc2(x)
        return x

t_net = TNet()

In [4]:
t_net

TNet(
  (fc1): Linear(in_features=5, out_features=4, bias=True)
  (norm): LayerNorm((4,), eps=1e-05, elementwise_affine=True)
  (fc2): Linear(in_features=4, out_features=3, bias=True)
)

## 1.2 创建LGflow模型
    通过LGflow，创建一个结构相同的模型。

In [5]:
class LNet(l_nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = l_nn.Linear(5, 4)
        self.norm = l_nn.LayerNorm(4)
        self.fc2 = l_nn.Linear(4, 3)

    def forward(self, x):
        x = self.fc1.forward(x)
        x = self.norm.forward(x)
        x = self.fc2.forward(x)
        return x

l_net = LNet()

In [6]:
print(l_net)

## 1.3 使LGflow模型的参数与torch模型参数相同

In [7]:
l_net.fc1.weights = l_nn.Parameter(t_net.fc1.weight.data.numpy())
l_net.fc1.bias = l_nn.Parameter(t_net.fc1.bias.data.numpy())
l_net.fc2.weights = l_nn.Parameter(t_net.fc2.weight.data.numpy())
l_net.fc2.bias = l_nn.Parameter(t_net.fc2.bias.data.numpy())

# 二. 创建输入

In [8]:
t_x = torch.randn(2, 5)    # input for torch
t_t = torch.tensor([[1, 0, 0], [0, 1, 0]], dtype=torch.float32)    # target for torch

In [9]:
t_x

tensor([[-1.4705, -0.6438, -0.6300,  1.7319,  0.2033],
        [-1.4013,  0.1755,  1.2352, -0.7821, -1.6463]])

In [10]:
l_x = LG_flow.Tensor(t_x.data.numpy())    # input for LGflow
l_t = LG_flow.Tensor([[1, 0, 0], [0, 1, 0]])    # target for LGflow

In [11]:
print(l_x)

(Tensor shape=(2, 5) dtype=float32 required_grad=False grad_fn=None 
[[-1.4704741  -0.6438229  -0.6300485   1.7319014   0.20330127]
 [-1.4013284   0.17552364  1.2351508  -0.7821366  -1.6462911 ]]
)


# 三. 创建损失函数

In [12]:
t_loss_fn = t_nn.CrossEntropyLoss(reduction='sum')

In [13]:
l_loss_fn = l_nn.CrossEntropyLoss(reduction='sum')

# 四. 创建优化器
    这里使用一个较大的学习率，使参数的更新幅度更大

In [14]:
t_optimizer = t_optim.SGD(t_net.parameters(), lr=0.1)

In [15]:
l_optimizer = l_optim.SGD(l_net.parameters(), lr=0.1)

# 五. 反向传播更新模型参数

    以epoch次反向更新为例，展示了LGflow与torch在链式求导与梯度更新方面具有一致性。

In [16]:
epochs = 20

In [17]:
for epoch in range(epochs):
    print('-'*50 + f'epoch: {epoch}' + '-'*50)
    
    # torch forward
    t_y = t_net.forward(t_x)
    print('t_y: ', t_y)

    # LGflow forward
    l_y = l_net.forward(l_x)
    print('l_y: ', l_y)
    print()
    
    # torch loss
    t_loss = t_loss_fn(t_y, t_t)
    print('t_loss: ', t_loss)

    # LGflow loss
    l_loss = l_loss_fn.forward(l_y, l_t)
    print('l_loss: ', l_loss)
    print()

    # torch backward
    t_optimizer.zero_grad()
    t_loss.backward()
    t_optimizer.step()
    
    # LGflow backward
    l_optimizer.zero_grad()
    l_loss.backward()
    l_optimizer.step()
    
    # check parameters
    ## torch parameters
    for param in t_net.parameters():
        print('torch model paramenters 0 value: ')
        print(param)
        print('torch model paramenters 0 grad: ')
        print(param.grad)
        break
        
    print()
    
    ## LGflow parameters
    for name, param in l_net.parameters().items():
        print('LGflow model paramenters 0 value: ')
        print(param)
        print('LGflow model paramenters 0 grad: ')
        print(param.grad)
        break


--------------------------------------------------epoch: 0--------------------------------------------------
t_y:  tensor([[-0.5223,  0.1089,  0.8782],
        [ 0.9183,  1.4450, -0.2491]], grad_fn=<AddmmBackward0>)
l_y:  (Tensor shape=(2, 3) dtype=float32 required_grad=False grad_fn=<LG_flow.Math_op.ADD_WITH_TENSOR object at 0x7bd6ef97efd0> 
[[-0.52234894  0.10889941  0.87823755]
 [ 0.9183017   1.4450424  -0.249075  ]]
)

t_loss:  tensor(2.5103, grad_fn=<NegBackward0>)
l_loss:  (Tensor shape=() dtype=float32 required_grad=False grad_fn=<LG_flow.Math_op.SUM object at 0x7bd6ef903490> 
2.510345935821533
)

torch model paramenters 0 value: 
Parameter containing:
tensor([[-0.2985, -0.1488,  0.1241,  0.1586,  0.1903],
        [-0.3269,  0.3566,  0.0725,  0.2258, -0.0284],
        [-0.2413,  0.2405,  0.1178,  0.2333, -0.2415],
        [-0.3762,  0.1390, -0.3112,  0.4176, -0.0238]], requires_grad=True)
torch model paramenters 0 grad: 
tensor([[-1.0053,  0.1458,  0.9323, -0.6223, -1.2274],
   